# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 7: Build a Dashboard with Plotly Dash

**Author:** Roberto Cortez  
**GitHub:** rtez-tech

This builds an interactive Plotly Dash application (`spacex_dash_app.py`) with:
- A dropdown to select a launch site (or all sites)
- A pie chart of successful launches
- A payload range slider
- A scatter plot of payload vs. launch outcome colored by booster version

> **Note:** Dash apps run as a standalone Python script, not inline in a notebook.
> Save the code below to `spacex_dash_app.py`, run `python3 spacex_dash_app.py`, open
> http://127.0.0.1:8050 in your browser, and take screenshots for slides 39–41.

In [3]:
# Save the Dash application to a file
dash_code = '''
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

spacex_df = pd.read_csv("spacex_launch_dash.csv")
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign':'center','color':'#503D36','font-size':40}),
    dcc.Dropdown(id='site-dropdown',
        options=[{'label':'All Sites','value':'ALL'}] +
                [{'label':s,'value':s} for s in spacex_df['Launch Site'].unique()],
        value='ALL', placeholder='Select a Launch Site', searchable=True),
    html.Br(),
    html.Div(dcc.Graph(id='success-pie-chart')),
    html.Br(),
    html.P("Payload range (Kg):"),
    dcc.RangeSlider(id='payload-slider', min=0, max=10000, step=1000,
        marks={0:'0',2500:'2500',5000:'5000',7500:'7500',10000:'10000'},
        value=[min_payload, max_payload]),
    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
])

@app.callback(Output('success-pie-chart','figure'), Input('site-dropdown','value'))
def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        fig = px.pie(spacex_df, values='class', names='Launch Site',
                     title='Total Successful Launches by Site')
    else:
        filtered = spacex_df[spacex_df['Launch Site']==entered_site]
        counts = filtered['class'].value_counts().reset_index()
        counts.columns = ['class','count']
        fig = px.pie(counts, values='count', names='class',
                     title=f'Total Success vs. Failure for {entered_site}')
    return fig

@app.callback(Output('success-payload-scatter-chart','figure'),
              [Input('site-dropdown','value'), Input('payload-slider','value')])
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range
    mask = (spacex_df['Payload Mass (kg)']>=low) & (spacex_df['Payload Mass (kg)']<=high)
    df_f = spacex_df[mask]
    if entered_site != 'ALL':
        df_f = df_f[df_f['Launch Site']==entered_site]
    fig = px.scatter(df_f, x='Payload Mass (kg)', y='class',
                     color='Booster Version Category',
                     title='Payload vs. Outcome')
    return fig

if __name__ == '__main__':
    app.run(debug=True)
'''
with open('spacex_dash_app.py','w') as f:
    f.write(dash_code)
print("Wrote spacex_dash_app.py")

Wrote spacex_dash_app.py


In [2]:
# Download the dashboard dataset (run once)
import requests
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
with open("spacex_launch_dash.csv","wb") as f:
    f.write(requests.get(url).content)
print("Downloaded spacex_launch_dash.csv")

/Users/rob/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Downloaded spacex_launch_dash.csv


### Run the dashboard
In a terminal:
```bash
python3 spacex_dash_app.py
```
Then open http://127.0.0.1:8050.

### Screenshots to capture (slides 39–41)
1. Pie chart of success counts for **all** sites.
2. Pie chart for the site with the **highest** success ratio (KSC LC-39A).
3. Payload vs. outcome scatter, observing which payload range and booster version
   have the highest success rate.

**Next:** Lab 8 — Predictive Analysis (Classification).